# M1  Escalabilidad Fuerte y Débil
## Sistema de Recomendación Paralelo para E-Commerce — RetailRocket Dataset
### Entrega 3  Eficiencia y Escalabilidad

Este notebook cubre el criterio **"Eficiencia o escalabilidad"** de M1 en la
Entrega 3:

- **Escalabilidad fuerte**: mismo dataset completo, variando 1/2/4/8 workers
  (reutiliza la metodología ya validada en la Entrega 2).
- **Escalabilidad débil**: el dataset crece proporcionalmente al número de
  workers (1 worker → N filas, 8 workers → 8N filas), midiendo si el tiempo
  se mantiene constante — que es el comportamiento ideal en este caso.


## 0. Configuración inicial

In [ ]:
# !pip install polars dask[dataframe] pandas psutil


In [ ]:
import sys
from pathlib import Path

RAIZ_PROYECTO = Path(r"E:\Ing ciencia de datos\Septimo cuatri\Computacion paralela\clase 15\Codigos")
sys.path.append(str(RAIZ_PROYECTO / "src"))

from benchmarks import (
    benchmark_etl_workers,             # Escalabilidad fuerte (ya validada en E2)
    benchmark_etl_escalabilidad_debil, # Escalabilidad débil (nueva, E3)
    guardar_resultados,
)

CARPETA_DATASETS = RAIZ_PROYECTO.parent / "Datasets"
CARPETA_RESULTADOS_E3 = RAIZ_PROYECTO / "resultados" / "entrega3"
CARPETA_TEMPORAL = RAIZ_PROYECTO / "datos" / "_tmp_escalabilidad"
CARPETA_RESULTADOS_E3.mkdir(parents=True, exist_ok=True)

ruta_eventos = CARPETA_DATASETS / "events.csv"
print(f"events.csv encontrado: {ruta_eventos.exists()}")


## 1. Escalabilidad fuerte

Mismo dataset completo (2.75M filas), variando el número de workers. Esta
metodología ya se validó en la Entrega 2 (`speedup_etl.csv`); se vuelve a
correr aquí para que quede como parte formal del entregable de escalabilidad
de Entrega 3, con el mismo protocolo de warm-up + repeticiones.


In [ ]:
resultado_fuerte = benchmark_etl_workers(
    ruta_eventos, lista_workers=[1, 2, 4, 8], n_repeticiones=3, warm_up=1
)
guardar_resultados(resultado_fuerte, "escalabilidad_fuerte.csv", CARPETA_RESULTADOS_E3)
resultado_fuerte


## 2. Escalabilidad débil

El dataset crece proporcionalmente al número de workers: con `filas_base`
filas para 1 worker, se usan `filas_base * workers` filas para cada
configuración. El comportamiento ideal es que el **tiempo se mantenga
constante**  si crece, significa que el overhead de coordinación crece más
rápido que la capacidad de cómputo agregada.


In [ ]:
resultado_debil = benchmark_etl_escalabilidad_debil(
    ruta_eventos,
    carpeta_temporal=CARPETA_TEMPORAL,
    filas_base=300_000,
    lista_workers=[1, 2, 4, 8],
    n_repeticiones=3,
    warm_up=1,
)
guardar_resultados(resultado_debil, "escalabilidad_debil.csv", CARPETA_RESULTADOS_E3)
resultado_debil


## 3. Visualización comparativa

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(resultado_fuerte["workers"], resultado_fuerte["speedup"], marker="o", color="#0F6E56", label="Speedup real")
axes[0].plot(resultado_fuerte["workers"], resultado_fuerte["workers"], linestyle="--", color="gray", label="Speedup ideal")
axes[0].set_title("Escalabilidad fuerte — Speedup"); axes[0].set_xlabel("Workers"); axes[0].set_ylabel("Speedup"); axes[0].legend()

axes[1].plot(resultado_debil["workers"], resultado_debil["media_s"], marker="o", color="#993C1D", label="Tiempo real")
axes[1].axhline(resultado_debil["media_s"].iloc[0], linestyle="--", color="gray", label="Tiempo ideal (constante)")
axes[1].set_title("Escalabilidad débil — Tiempo"); axes[1].set_xlabel("Workers (y datos proporcionales)"); axes[1].set_ylabel("Tiempo (s)"); axes[1].legend()

plt.tight_layout()
plt.savefig(CARPETA_RESULTADOS_E3 / "escalabilidad_fuerte_debil.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Interpretación

**Escalabilidad fuerte:** ver el techo de speedup ya documentado en la
Entrega 2 (Amdahl, fracción secuencial ~54%). Se reafirma aquí sobre el
dataset completo con el protocolo formal de Entrega 3.

**Escalabilidad débil:** si `eficiencia_debil` se mantiene cercana a 1.0 en
todas las configuraciones, el sistema escala bien ante crecimiento de datos
proporcional a los recursos  un requisito más realista para el escenario de
"pico de tráfico tipo Black Friday" que exige el proyecto, donde tanto los
datos como (potencialmente) los recursos de cómputo crecen juntos.

*(Completar con la interpretación específica de los números obtenidos al
correr contra el dataset real completo.)*
